In [2]:
import json
import os
import pandas as pd
import glob
from transformers import AutoTokenizer
import re
import json
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, util
import torch
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))
from src.utils.eval_utils import calculate_metrics, parse_absa_string, parse_aoste

In [3]:
# model_name = "all-MiniLM-L6-v2"
model_name = "Qwen/Qwen3-Embedding-8B"
model = SentenceTransformer(
    model_name,
    trust_remote_code=True,
	model_kwargs={
		"device_map": "cuda",         # Force direct load to GPU
		"torch_dtype": torch.bfloat16, # Use FP16 (16GB VRAM) instead of FP32 (32GB)
		# "low_cpu_mem_usage": True,    # Critical: Prevents loading full model to RAM
		# "offload_folder": "offload"   # Safety net: Uses disk if RAM spikes
	},
	cache_folder=os.getenv('CACHE_DIR')
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.31s/it]


In [4]:
from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
	"""
	Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
	Each dictionary contains the tag as the key and the corresponding value.
	For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
	[{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
	{'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

	Args:
		text (str): ABSA string output to be parsed.

	Returns:
		List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

	"""
	pattern = r"\[(\w+)\]\s*([^[]+)"
	matches = re.findall(pattern, text)

	result = []
	current_dict = {}

	for tag, content in matches:
		if tag == "SSEP":  # Sentence separator -> Start a new dictionary
			result.append(current_dict)
			current_dict = {}
		else:
			current_dict[tag] = content.strip()

	if current_dict:  # Append the last sentence if it exists
		result.append(current_dict)

	return result

def parse_aoste(text: str) -> List[Dict[str, str]]:
	"""
	Format this: check in:dari jam 8 malam cek out jam 11 malam . karena tidak ada air:negative, wc:kotor:negative
	To this: [{"A": "check in", "O": "dari jam 8 malam cek out jam 11 malam . karena tidak ada air", "S": "negative"},
			  {"A": "wc", "O": "kotor", "S": "negative"}]
	Args:
		text (str): AOSTE string output to be parsed.

	Returns:
		List[Dict[str, str]]: List of dictionaries of parsed AOSTE output.

	"""
	pattern = r"([^:]+):([^,]+):(\w+)"
	matches = re.findall(pattern, text)

	result = []
	for aspect, opinion, sentiment in matches:
		result.append({
			"A": aspect.strip(),
			"O": opinion.strip(),
			"S": sentiment.strip()
		})

	return result

In [5]:
instruction1 = "Retrieve semantically similar text."
instruction2 = "Retrieve semantically similar text. If a text is somewhat a subset of the other text, it should still be considered similar as long as it is not contradictive."
instruction3 = """Retrieve semantically similar text. If a text is somewhat a subset of the other text, it should still be considered similar as long as it is not contradictive.
Example 1:
Sentence A: [A] harga [O] pas [S] positive
Sentence B: [A] harga [O] pas di kantong [S] positive
Similarity: High

Example 2:
Sentence A: [A] budget [O] sesuai [S] positive
Sentence B: [A] null [O] sesuai budget [S] positive
Similarity: High

Example 3:
Sentence A: [A] lift nya [O] di perbaiki . karena sudah kelihatan sudah lama sekali [S] negative
Sentence B: [A] lift nya [O] kelihatan [S] negative
Similarity: Medium
"""
instruction4 = """Retrieve semantically similar text.
The text that will be retrieved are a tuple of Aspect-Based Sentiment Analysis task containing the aspect term, opinion term, and sentiment with format "[A] aspect term [O] opinion term [S] sentiment".
The aspect and opinion can be a subset of the other text as long as it is not contradictive.
The aspect can be null if there is no aspect term (implicit aspect), but the opinion term must exist.
The sentiment should be the same for both texts.
"""

In [6]:
test_cases = [
    {
        "truth": "[A] harga [O] pas [S] positive",
        "preds": [
            "[A] harga [O] pas di kantong [S] positive"
        ],
        "wrong_preds": [
            "[A] harga [O] terlalu mahal [S] negative"
        ]
    },
    {
        "truth": "[A] budget [O] sesuai [S] positive",
        "preds": [
            "[A] null [O] sesuai budget [S] positive"
        ],
        "wrong_preds": [
            "[A] lokasi [O] sangat jauh [S] negative"
        ]
    },
    {
        "truth": "[A] antar jemput [O] disediakan [S] positive",
        "preds": [
            "[A] null [O] disediakan antar jemput ke dan dari bandara [S] positive"
        ],
        "wrong_preds": [
            "[A] parkir [O] tidak tersedia [S] negative",
            "[A] null [O] tidak disediakan antar jemput ke dan dari bandara [S] positive"
        ]
    },
    {
        "truth": "[A] ruangan [O] terlalu sempit [S] negative",
        "preds": [
            "[A] null [O] hanya saja ruangan terlalu sempit [S] negative"
        ],
        "wrong_preds": [
            "[A] ruangan [O] sangat luas dan lega [S] positive"
        ]
    },
    {
        "truth": "[A] breakfast [O] kuah juga keasinan [S] negative",
        "preds": [
            "[A] null [O] kuah juga keasinan [S] negative"
        ],
        "wrong_preds": [
            "[A] dessert [O] manis sekali [S] positive"
        ]
    },
    {
        "truth": "[A] null [O] dilayani dengan baik [S] positive",
        "preds": [
            "[A] dilayani [O] dengan baik [S] positive"
        ],
        "wrong_preds": [
            "[A] pelayanan [O] sangat buruk dan lambat [S] negative"
        ]
    },
    {
        "truth": "[A] null [O] nyaman stay disini [S] positive",
        "preds": [
            "[A] null [O] nyaman [S] positive"
        ],
        "wrong_preds": [
            "[A] kebersihan [O] lantai kotor [S] negative"
        ]
    },
    {
        "truth": "[A] mas yang jaga malam [O] baik [S] positive",
        "preds": [
            "[A] stay [O] disini mas yang jaga malam baik [S] positive"
        ],
        "wrong_preds": [
            "[A] security [O] galak dan tidak sopan [S] negative"
        ]
    },
    {
        "truth": "[A] air [O] check in dari jam 8 malam cek out jam 11 malam . karena tidak ada air [S] negative",
        "preds": [
            "[A] air [O] tidak ada air [S] negative"
        ],
        "wrong_preds": [
            "[A] air [O] mengalir deras dan bersih [S] positive"
        ]
    },
    {
        "truth": "[A] lift nya [O] di perbaiki . karena sudah kelihatan sudah lama sekali [S] negative",
        "preds": [
            "[A] lift nya [O] kelihatan [S] negative",
            "[A] lift nya [O] sudah lama sekali [S] negative",
            "[A] null [O] cukup [S] positive"
        ],
        "wrong_preds": [
            "[A] lift [O] baru dan canggih [S] positive"
        ]
    },
    {
        "truth": "[A] ac [O] mati setiap 30 menit sekali ( tidak bisa direset timernya ) [S] negative",
        "preds": [
            "[A] ac [O] mati setiap 30 menit sekali [S] negative",
            "[A] ac [O] tidak bisa direset timernya [S] negative"
        ],
        "wrong_preds": [
            "[A] ac [O] sangat dingin dan nyaman [S] positive"
        ]
    },
    {
        "truth": "[A] breakfast [O] soto madura sudah tidak ada dagingnya sama sekali [S] negative",
        "preds": [
            "[A] dagingnya [O] tidak ada dagingnya sama sekali [S] negative",
            "[A] dagingnya [O] ada dagingnya sama sekali [S] negative"
        ],
        "wrong_preds": [
            "[A] makanan [O] porsi daging melimpah [S] positive"
        ]
    },
    {
        "truth": "[A] resepsionis [O] perlu di perbaiki supaya lebih ramah dan sopan [S] negative",
        "preds": [
            "[A] resepsionis [O] perlu di perbaiki [S] negative",
            "[A] resepsionis [O] lebih ramah [S] negative",
            "[A] resepsionis [O] sopan [S] negative"
        ],
        "wrong_preds": [
            "[A] resepsionis [O] sangat murah senyum [S] positive"
        ]
    },
    {
        "truth": "[A] kolam renangnya [O] ada kolam renangnya yang buat anak senang [S] positive",
        "preds": [
            "[A] kolam renangnya [O] buat anak senang [S] positive"
        ],
        "wrong_preds": [
            "[A] fasilitas [O] tidak ada kolam renang [S] negative"
        ]
    },
    {
        "truth": "[A] kursi lipat [O] ada [S] positive",
        "preds": [
            "[A] kursi [O] lipat [S] positive"
        ],
        "wrong_preds": [
            "[A] meja [O] tidak tersedia [S] negative"
        ]
    },
    {
        "truth": "[A] wifi [O] gratis wifi katanya ternyata tidak dikasih passwordnya aneh [S] negative",
        "preds": [
            "[A] wifi [O] tidak dikasih passwordnya aneh [S] negative"
        ],
        "wrong_preds": [
            "[A] internet [O] koneksi sangat cepat [S] positive"
        ]
    }
]

In [7]:
def format_sts(text, instruction):
	return f"Instruct: {instruction}\nQuery: {text}"

for test_case in test_cases:
    print("========================================")
    sentence_truth = test_case["truth"]
    sentence_preds = test_case["preds"]
    sentence_wrong = test_case["wrong_preds"]


    emb_truth = model.encode(format_sts(sentence_truth, instruction1))
    emb_preds = []
    wrong_embs = []

    for pred in sentence_preds:
        emb_preds.append(model.encode(format_sts(pred, instruction1)))

    for pred in sentence_wrong:
        wrong_embs.append(model.encode(format_sts(pred, instruction1)))



    scores = []
    for i, emb_pred in enumerate(emb_preds):
        score = util.cos_sim(emb_truth, emb_pred)
        scores.append((sentence_preds[i], score))

    wrong_scores = []
    for i, emb_pred in enumerate(wrong_embs):
        score = util.cos_sim(emb_truth, emb_pred)
        wrong_scores.append((sentence_wrong[i], score))


    print(f"Instruction: {instruction1}")
    print(f"Ground truth: {sentence_truth}\n")
    for i, score in enumerate(scores):
        print(f"Sentence {i}: {score[0]}\nSimilarity: {score[1].item():.4f}\n")
    print("------Wrong sentences------")
    for i, score in enumerate(wrong_scores):
        print(f"Sentence {i}: {score[0]}\nSimilarity: {score[1].item():.4f}\n")

Instruction: Retrieve semantically similar text.
Ground truth: [A] harga [O] pas [S] positive

Sentence 0: [A] harga [O] pas di kantong [S] positive
Similarity: 0.9684

------Wrong sentences------
Sentence 0: [A] harga [O] terlalu mahal [S] negative
Similarity: 0.8339

Instruction: Retrieve semantically similar text.
Ground truth: [A] budget [O] sesuai [S] positive

Sentence 0: [A] null [O] sesuai budget [S] positive
Similarity: 0.9729

------Wrong sentences------
Sentence 0: [A] lokasi [O] sangat jauh [S] negative
Similarity: 0.7180

Instruction: Retrieve semantically similar text.
Ground truth: [A] antar jemput [O] disediakan [S] positive

Sentence 0: [A] null [O] disediakan antar jemput ke dan dari bandara [S] positive
Similarity: 0.9165

------Wrong sentences------
Sentence 0: [A] parkir [O] tidak tersedia [S] negative
Similarity: 0.7619

Sentence 1: [A] null [O] tidak disediakan antar jemput ke dan dari bandara [S] positive
Similarity: 0.8145

Instruction: Retrieve semantically si

In [8]:
data_paths = glob.glob('outputs/evals/hotel_reviews/indo/mvp_aos/seed_*/*/*/*/*/inference_results.json')
# data_paths += glob.glob('outputs/evals/hotel_reviews/indo/mvp/seed_*/*/*/*/*/voting_results.json')
data_paths

['outputs/evals/hotel_reviews/indo/mvp_aos/seed_2024/20260226_063347_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-1560/checkpoint-1560/constrained_decoding/inference_results.json',
 'outputs/evals/hotel_reviews/indo/mvp_aos/seed_2024/20260226_063347_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-1560/checkpoint-1560/unconstrained_decoding/inference_results.json',
 'outputs/evals/hotel_reviews/indo/mvp_aos/seed_31415/20260226_065122_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-1560/checkpoint-1560/constrained_decoding/inference_results.json',
 'outputs/evals/hotel_reviews/indo/mvp_aos/seed_31415/20260226_065122_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-1560/checkpoint-1560/unconstrained_decoding/inference_results.json',
 'outputs/evals/hotel_reviews/indo/mvp_aos/seed_777/20260226_070836_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-1560/checkpoint-1560/constrained_decoding/inference_results.json',
 'outputs/evals

In [9]:
from tqdm import tqdm
def calculate_metrics_semantic(predictions: List[List[Dict[str, str]]], targets: List[List[Dict[str, str]]], model: SentenceTransformer, task='') -> Dict[str, float]:
	"""
	Calculate precision, recall, and F1 score for the given predictions and targets for ABSA.

	Args:
		predictions (List[List[Dict[str, str]]]): List of predicted triplets.
		targets (List[List[Dict[str, str]]]): List of target triplets.
		model (SentenceTransformer): The sentence transformer model to use for embedding similarity.
		task (str): The task name for which metrics are calculated.
	
	Returns:
		Dict[str, float]: A dictionary containing precision, recall, and F1 score.
	"""
	true_positive = 0
	false_positive = 0
	false_negative = 0
	for prediction,target in tqdm(zip(predictions,targets), total=len(predictions), desc=f"Calculating semantic metrics"):
		false_negative_candidates = []
		false_positive_candidates = []
		for target_tuple in target:
			if target_tuple in prediction:
				true_positive += 1
			else:
				false_negative_candidates.append(target_tuple)
				# false_negative += 1
		false_positive_candidates += [pred for pred in prediction if pred not in target]
		# false_positive += sum(1 for pred in prediction if pred not in target)
	
		# Check candidate with embedding similarity
		# For each false negative candidate, check if there's a similar prediction in false_positive_candidates
		# Threshold is 0.9
		# print(f"False negative candidates: {false_negative_candidates}")
		for false_negative_candidate in false_negative_candidates:
			emb_false_negative = model.encode(format_sts(str(false_negative_candidate), instruction4))
			for false_positive_candidate in false_positive_candidates:
				emb_false_positive = model.encode(format_sts(str(false_positive_candidate), instruction4))
				score = util.cos_sim(emb_false_negative, emb_false_positive)
				# print(f"Comparing false negative candidate: {false_negative_candidate} with false positive candidate: {false_positive_candidate}, score: {score.item()}")
				if score.item() >= 0.9:
					# print(f"Found a match!")
					true_positive += 1
					false_positive_candidates.remove(false_positive_candidate)
					break
			else:
				false_negative += 1
		false_positive += len(false_positive_candidates)

	# Calculate final metrics
	precision = true_positive/(true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
	recall = true_positive/(true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
	f1 = (2 * recall * precision)/(recall + precision) if (recall + precision) > 0 else 0
	return {
		f"precision_{task}" : precision,
		f"recall_{task}" : recall,
		f"f1_{task}" : f1
	}

In [10]:
data_results = {
    'lang': [],
    'seed': [],
    'use_constrained_decoding': [],
    'precision': [],
    'recall': [],
    'f1': []
}

In [11]:
# ['outputs/evals/hotel_reviews/indo/mvp_aos/seed_2024/20260226_063347_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-1560/checkpoint-1560/constrained_decoding/inference_results.json',


In [12]:
for path in data_paths:
	lang = path.split('/')[3]
	seed = path.split('/')[5].split('_')[1]
	dataset_folder = path.split('/')[4]
	dataset_type = path.split('/')[2]
	use_constrained_decoding = path.split('/')[-2] == 'constrained_decoding'
	print(f"Processing lang: {lang}, seed: {seed}, dataset_folder: {dataset_folder}, dataset_type: {dataset_type}, use_constrained_decoding: {use_constrained_decoding}")
	with open(path, 'r') as f:
		data = json.load(f)
	target_lists = [item['target_list'] for item in data]
	prediction_lists = [item['prediction_list'] for item in data]
	scores = calculate_metrics_semantic(prediction_lists, target_lists, model, task='semantic')
	data_results['lang'].append(lang)
	data_results['seed'].append(seed)
	data_results['use_constrained_decoding'].append(use_constrained_decoding)
	data_results['precision'].append(scores['precision_semantic'])
	data_results['recall'].append(scores['recall_semantic'])
	data_results['f1'].append(scores['f1_semantic'])


Processing lang: indo, seed: 2024, dataset_folder: mvp_aos, dataset_type: hotel_reviews, use_constrained_decoding: True


Calculating semantic metrics: 100%|██████████| 1000/1000 [02:06<00:00,  7.92it/s]


Processing lang: indo, seed: 2024, dataset_folder: mvp_aos, dataset_type: hotel_reviews, use_constrained_decoding: False


Calculating semantic metrics: 100%|██████████| 1000/1000 [02:07<00:00,  7.85it/s]


Processing lang: indo, seed: 31415, dataset_folder: mvp_aos, dataset_type: hotel_reviews, use_constrained_decoding: True


...

Calculating semantic metrics: 100%|██████████| 1000/1000 [03:02<00:00,  5.47it/s]


Processing lang: indo, seed: 123, dataset_folder: mvp_aos, dataset_type: hotel_reviews, use_constrained_decoding: True


Calculating semantic metrics: 100%|██████████| 1000/1000 [02:54<00:00,  5.74it/s]


Processing lang: indo, seed: 123, dataset_folder: mvp_aos, dataset_type: hotel_reviews, use_constrained_decoding: False


Calculating semantic metrics: 100%|██████████| 1000/1000 [02:56<00:00,  5.66it/s]


In [13]:
df = pd.DataFrame(data_results)
df.to_csv('semantic_similarity_metrics_novoting.csv', index=False)

In [55]:
target_lists = [item['target_list'] for item in data]
prediction_lists = [item['prediction_list'] for item in data]
calculate_metrics(prediction_lists, target_lists)

{'precision_': 0.6975100942126514,
 'recall_': 0.6942397856664434,
 'f1_': 0.6958710976837865}

In [56]:
target_lists = [item['target_list'] for item in data]
prediction_lists = [item['prediction_list'] for item in data]
calculate_metrics_semantic(prediction_lists, target_lists, model, task='semantic')

Calculating semantic metrics: 100%|██████████| 1000/1000 [02:03<00:00,  8.11it/s]


{'precision_semantic': 0.8596904441453567,
 'recall_semantic': 0.8556597454789016,
 'f1_semantic': 0.8576703591809332}